# Configuracion del proyecto y base vectorial local

Este notebook prepara los recursos del proyecto y crea una base ChromaDB local con dos colecciones:

- `restaurant_articles`: documentos textuales de restaurantes.
- `food_images`: documentos asociados a imagenes de recetas.

**Ejecutar este notebook completo desde la raiz de este repositorio antes de abrir los notebooks posteriores.**
Las carpetas `data/`, `recipe_images/` y `datadb/` se crean automaticamente en esa raiz.

## Requisitos

Antes de ejecutar el notebook, colocar en la raiz del repositorio estos archivos de entrada:

- `structured_restaurant_data.json`
- `augmented_food_recipe.json`
- `synthetic-recipe-images.zip` (opcional; si falta, se descarga automaticamente)
- `structured-restaurant-data.json`
- `augmented-user-review.json`
- `California-Culinary-Map.txt`

El notebook copiara los datos JSON y el ZIP a `data/`, extraera las imagenes en `recipe_images/` y persistira la base vectorial en `datadb/`. Estas tres carpetas son locales y estan excluidas de Git.

In [ ]:
# Ejecutar una vez para instalar todas las dependencias del proyecto.
%pip install -r requirements.txt

In [ ]:
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from pathlib import Path
import json
import shutil
import urllib.request
import zipfile

import numpy as np
import torch
from PIL import Image
from langchain_chroma import Chroma
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer
from transformers import CLIPModel, CLIPProcessor
from transformers.utils.logging import disable_progress_bar

disable_progress_bar()

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
IMAGE_DIR = PROJECT_DIR / "recipe_images"
DB_DIR = PROJECT_DIR / "datadb"
SOURCE_DATA_DIR = PROJECT_DIR
SOURCE_FILES = {
    "structured_restaurant_data.json": SOURCE_DATA_DIR / "structured_restaurant_data.json",
    "augmented_food_recipe.json": SOURCE_DATA_DIR / "augmented_food_recipe.json",
    "synthetic-recipe-images.zip": SOURCE_DATA_DIR / "synthetic-recipe-images.zip",
    "structured-restaurant-data.json": SOURCE_DATA_DIR / "structured-restaurant-data.json",
    "augmented-user-review.json": SOURCE_DATA_DIR / "augmented-user-review.json",
    "California-Culinary-Map.txt": SOURCE_DATA_DIR / "California-Culinary-Map.txt",
}

DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
DB_DIR.mkdir(parents=True, exist_ok=True)
print(f"Raiz del proyecto: {PROJECT_DIR}")
print(f"Base vectorial: {DB_DIR}")

In [ ]:
# Copiar los recursos de entrada a data/ y obtener el ZIP de imagenes si hace falta.
IMAGE_ZIP_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/5_Rr6ohviItzucyWk6nkrw/synthetic-recipe-images.zip"

for filename in (
    "structured_restaurant_data.json",
    "augmented_food_recipe.json",
    "structured-restaurant-data.json",
    "augmented-user-review.json",
    "California-Culinary-Map.txt",
):
    destination = DATA_DIR / filename
    source = destination if destination.exists() else PROJECT_DIR / filename
    if not source.exists():
        raise FileNotFoundError(
            f"No se encontro {filename}. Colocalo en la raiz del repositorio "
            f"o en la carpeta data/: {destination}"
        )
    if source.resolve() != destination.resolve():
        shutil.copy2(source, destination)

zip_path = DATA_DIR / "synthetic-recipe-images.zip"
if not zip_path.exists():
    source_zip = PROJECT_DIR / "synthetic-recipe-images.zip"
    if source_zip.exists():
        shutil.copy2(source_zip, zip_path)
    else:
        urllib.request.urlretrieve(IMAGE_ZIP_URL, zip_path)

if not any(IMAGE_DIR.rglob("*.png")):
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(IMAGE_DIR)

image_paths = sorted(IMAGE_DIR.rglob("*.png"))
print(f"JSON preparados: {len(list(DATA_DIR.glob('*.json')))}")
print(f"Imagenes encontradas: {len(image_paths)}")

In [ ]:
with open(DATA_DIR / "structured_restaurant_data.json", encoding="utf-8") as file:
    restaurants = json.load(file)
with open(DATA_DIR / "augmented_food_recipe.json", encoding="utf-8") as file:
    recipes = json.load(file)

text_model = SentenceTransformer("all-MiniLM-L6-v2")
device = "cpu"
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", use_fast=True)
clip_model.eval()

def embed_texts(texts, batch_size=64):
    return text_model.encode(
        texts, batch_size=batch_size, show_progress_bar=False,
        normalize_embeddings=True,
    ).astype(np.float32)

@torch.no_grad()
def embed_images(paths, batch_size=16):
    vectors = []
    for start in range(0, len(paths), batch_size):
        batch = paths[start:start + batch_size]
        images = []
        try:
            for path in batch:
                with Image.open(path) as image:
                    images.append(image.convert("RGB"))
            inputs = clip_processor(images=images, return_tensors="pt").to(device)
            features = clip_model.get_image_features(**inputs)
            features = features / features.norm(dim=-1, keepdim=True)
            vectors.append(features.cpu().numpy().astype(np.float32))
            del inputs, features
        finally:
            images.clear()
    return np.vstack(vectors)

article_docs = []
for index, restaurant in enumerate(restaurants):
    name = str(restaurant.get("name", "")).strip()
    if not name:
        continue
    content = (
        f"Restaurant: {name}\n"
        f"Cuisine: {restaurant.get('food_style', '')}\n"
        f"Location: {restaurant.get('location', '')}"
    )
    article_docs.append(Document(
        page_content=content,
        metadata={
            "doc_id": f"rest_{index}",
            "cuisine": restaurant.get("food_style"),
            "location": restaurant.get("location"),
            "source": "restaurant",
        },
    ))

image_docs = []
for index, (path, recipe) in enumerate(zip(image_paths, recipes)):
    image_docs.append(Document(
        page_content=recipe.get("name", f"recipe image {index}"),
        metadata={
            "doc_id": f"img_{index}",
            "image_path": str(path),
            "source": "recipe_image",
            "recipe_id": recipe.get("id"),
            "cuisine": recipe.get("cuisine"),
        },
    ))

print(f"Documentos de restaurantes: {len(article_docs)}")
print(f"Documentos de imagenes: {len(image_docs)}")

In [ ]:
# Crear o reemplazar la base local del proyecto.
if DB_DIR.exists():
    shutil.rmtree(DB_DIR)
DB_DIR.mkdir(parents=True, exist_ok=True)

article_vectors = embed_texts([doc.page_content for doc in article_docs])
article_db = Chroma(
    collection_name="restaurant_articles",
    persist_directory=str(DB_DIR),
)
article_db._collection.upsert(
    ids=[doc.metadata["doc_id"] for doc in article_docs],
    embeddings=article_vectors.tolist(),
    documents=[doc.page_content for doc in article_docs],
    metadatas=[doc.metadata for doc in article_docs],
)

image_vectors = embed_images([doc.metadata["image_path"] for doc in image_docs])
image_db = Chroma(
    collection_name="food_images",
    persist_directory=str(DB_DIR),
)
image_db._collection.upsert(
    ids=[doc.metadata["doc_id"] for doc in image_docs],
    embeddings=image_vectors.tolist(),
    documents=[doc.page_content for doc in image_docs],
    metadatas=[doc.metadata for doc in image_docs],
)

print(f"Coleccion de restaurantes: {article_db._collection.count()} registros")
print(f"Coleccion de imagenes: {image_db._collection.count()} registros")
print(f"Base persistida en: {DB_DIR}")
print("Multimodal Vector Index Construction COMPLETE")

## Uso desde otros notebooks

Los notebooks posteriores deben abrir las colecciones usando la misma ruta desde la raiz del repositorio:

```python
from pathlib import Path

DB_DIR = Path.cwd() / "datadb"
article_db = Chroma(collection_name="restaurant_articles", persist_directory=str(DB_DIR))
image_db = Chroma(collection_name="food_images", persist_directory=str(DB_DIR))
```

Ejecutar primero este notebook completo desde la raiz del repositorio. Despues se puede ejecutar los notebooks de recuperacion.